# 2. Polynomial Interpolation

Given data points $(x_i, y_i)$, find a polynomial passing through all of them. This notebook covers:
- **Lagrange** interpolation
- **Newton's divided differences**
- **Cubic spline** interpolation
- Runge phenomenon and comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, lagrange

%matplotlib inline

## 2.1 Lagrange Interpolation

$$P(x) = \sum_{i=0}^{n} y_i \prod_{j \neq i} \frac{x - x_j}{x_i - x_j}$$

Each Lagrange basis polynomial $L_i(x)$ equals 1 at $x_i$ and 0 at all other nodes.

In [ ]:
def lagrange_interp(x_nodes, y_nodes, x_eval):
    """Evaluate Lagrange interpolant at points x_eval."""
    n = len(x_nodes)
    result = np.zeros_like(x_eval, dtype=float)
    for i in range(n):
        Li = np.ones_like(x_eval, dtype=float)
        for j in range(n):
            if j != i:
                Li *= (x_eval - x_nodes[j]) / (x_nodes[i] - x_nodes[j])
        result += y_nodes[i] * Li
    return result

# Example: interpolate sin(x)
x_nodes = np.array([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
y_nodes = np.sin(x_nodes)

x_fine = np.linspace(0, np.pi, 200)
y_lagrange = lagrange_interp(x_nodes, y_nodes, x_fine)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_fine, np.sin(x_fine), 'k-', lw=2, label='sin(x) (exact)')
ax.plot(x_fine, y_lagrange, 'r--', lw=2, label='Lagrange interpolant')
ax.plot(x_nodes, y_nodes, 'ko', markersize=8, label='Nodes')
ax.set_title('Lagrange Interpolation of sin(x)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.2 Newton's Divided Differences

$$P(x) = f[x_0] + f[x_0, x_1](x-x_0) + f[x_0, x_1, x_2](x-x_0)(x-x_1) + \cdots$$

More efficient for adding new points incrementally.

In [ ]:
def divided_diff(x, y):
    """Compute the divided difference table."""
    n = len(x)
    table = np.zeros((n, n))
    table[:, 0] = y
    for j in range(1, n):
        for i in range(n - j):
            table[i, j] = (table[i+1, j-1] - table[i, j-1]) / (x[i+j] - x[i])
    return table[0, :]  # top row = coefficients

def newton_interp(x_nodes, coeffs, x_eval):
    """Evaluate Newton interpolant using Horner-like scheme."""
    n = len(coeffs)
    result = coeffs[-1] * np.ones_like(x_eval, dtype=float)
    for i in range(n-2, -1, -1):
        result = result * (x_eval - x_nodes[i]) + coeffs[i]
    return result

coeffs = divided_diff(x_nodes, y_nodes)
y_newton = newton_interp(x_nodes, coeffs, x_fine)

print("Divided difference coefficients:", np.round(coeffs, 6))
print(f"Max difference Lagrange vs Newton: {np.max(np.abs(y_lagrange - y_newton)):.2e}")

## 2.3 Runge Phenomenon

High-degree polynomial interpolation on equally-spaced nodes can oscillate wildly near the boundaries. The classic example is the **Runge function** $f(x) = \frac{1}{1 + 25x^2}$.

In [ ]:
runge = lambda x: 1 / (1 + 25 * x**2)
x_fine_r = np.linspace(-1, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, n_pts, title in zip(axes, [11, 21], ['n = 11', 'n = 21']):
    x_eq = np.linspace(-1, 1, n_pts)
    y_eq = runge(x_eq)
    p = lagrange_interp(x_eq, y_eq, x_fine_r)
    ax.plot(x_fine_r, runge(x_fine_r), 'k-', lw=2, label='Runge function')
    ax.plot(x_fine_r, p, 'r--', lw=1.5, label=f'Lagrange (deg {n_pts-1})')
    ax.plot(x_eq, y_eq, 'ko', markersize=4)
    ax.set_ylim(-1, 2)
    ax.set_title(f'Runge Phenomenon ({title})')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.4 Cubic Spline Interpolation

Instead of one high-degree polynomial, use piecewise cubics joined smoothly ($C^2$ continuity). This avoids the Runge phenomenon.

In [ ]:
n_pts = 11
x_eq = np.linspace(-1, 1, n_pts)
y_eq = runge(x_eq)

cs = CubicSpline(x_eq, y_eq)
y_spline = cs(x_fine_r)
y_lag = lagrange_interp(x_eq, y_eq, x_fine_r)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_fine_r, runge(x_fine_r), 'k-', lw=2, label='Runge function')
ax.plot(x_fine_r, y_lag, 'r--', lw=1.5, alpha=0.7, label='Lagrange (deg 10)')
ax.plot(x_fine_r, y_spline, 'b-', lw=2, label='Cubic Spline')
ax.plot(x_eq, y_eq, 'ko', markersize=5)
ax.set_ylim(-0.5, 1.5)
ax.set_title('Cubic Spline vs Lagrange on Runge Function')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max error Lagrange: {np.max(np.abs(runge(x_fine_r) - y_lag)):.4f}")
print(f"Max error Spline:   {np.max(np.abs(runge(x_fine_r) - y_spline)):.4f}")

## Key Takeaways

- **Lagrange** and **Newton** forms produce the same unique polynomial but differ computationally
- High-degree polynomial interpolation on uniform nodes suffers from the **Runge phenomenon**
- **Cubic splines** avoid oscillation by using piecewise low-degree polynomials
- For equally-spaced nodes, prefer splines; for optimal nodes, use **Chebyshev points**